### Silver - Produtos

Problemas identificados:
- `product_id` em caixa baixa em diversos registros (`p0013`, `p0026`...),
  o que impede o join com os itens de pedido, que utilizam sempre caixa
  alta
- `P0006` duplicado no próprio cadastro
- `status` com valores `Ativo`/`ativo`/`inativo`/`descontinuado`/null

O código `P8888` é referenciado em itens de pedido mas não possui
registro correspondente no cadastro, mesmo após normalização de caixa
(diferente de `P9999`, que existe no cadastro). Esse caso não é tratado
nesta dimensão; a resolução ocorre na camada de fato, via left join com
uma flag de produto não cadastrado.

In [ ]:
%run "../00_setup/00_setup_ambiente"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_produtos_bronze = spark.table(f"{schema_name}.bronze_produtos")

In [ ]:
df_produtos_flat = df_produtos_bronze.select(
    F.upper(F.trim(F.col("product.product_id"))).alias("product_id"),
    F.col("product.name").alias("nome_produto"),
    F.col("product.category").alias("categoria"),
    F.col("product.subcategory").alias("subcategoria"),
    F.col("product.status").alias("status_raw"),
    F.col("pricing.list_price").cast("string").alias("preco_lista_raw"),
    F.col("pricing.currency").alias("moeda"),
    F.col("attributes.family").alias("familia"),
    F.col("attributes.tags").alias("tags"),
    F.expr("try_to_timestamp(updated_at)").alias("updated_at"),
)

O campo pricing.list_price apresenta tipos mistos no JSON de origem: a
maioria dos valores é numérica, mas há o literal 'N/A' (produto P9999) e
valores numéricos representados como string entre aspas. A inferência de
schema do Spark na leitura do JSON pode não capturar essa inconsistência
de forma confiável entre ambientes, gerando falha de cast apenas na
escrita da camada Gold. A coluna é lida como string e convertida com
try_cast para double, retornando null para valores não numéricos.

In [ ]:
df_produtos_flat = df_produtos_flat.withColumn(
    "preco_lista", F.expr("try_cast(preco_lista_raw AS DOUBLE)")
).drop("preco_lista_raw")

df_produtos_status = df_produtos_flat.withColumn(
    "status_produto",
    F.when(F.lower(F.col("status_raw")) == "ativo", "Ativo")
     .when(F.lower(F.col("status_raw")) == "inativo", "Inativo")
     .when(F.lower(F.col("status_raw")) == "descontinuado", "Descontinuado")
     .otherwise("Não informado")
).drop("status_raw")

In [ ]:
# dedup por updated_at mais recente
w_dedup = Window.partitionBy("product_id").orderBy(F.col("updated_at").desc())

df_produtos_silver = (
    df_produtos_status
    .withColumn("rk", F.row_number().over(w_dedup))
    .filter(F.col("rk") == 1)
    .drop("rk")
)

(
    df_produtos_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.silver_produtos")
)

print("bronze:", df_produtos_bronze.count(), "-> silver:", df_produtos_silver.count())

Identificação de produtos órfãos (sem alteração de dados, apenas
verificação para documentação):

In [ ]:
df_itens_bronze = spark.table(f"{schema_name}.bronze_pedidos_itens")

produtos_em_itens = df_itens_bronze.select(F.upper(F.trim(F.col("product_code"))).alias("product_id")).distinct()
produtos_cadastrados = df_produtos_silver.select("product_id").distinct()

df_orfaos = produtos_em_itens.join(produtos_cadastrados, "product_id", "left_anti")
display(df_orfaos)